# flan-t5 Fine-Tuning: Car Rental Review Classification & Sentiment

Fine-tunes `google/flan-t5-large` on real Floyt car rental reviews (DE/FR) using LoRA adapters.

The model is trained on three tasks simultaneously:
- **Classification** — assign 1–3 categories (Vehicle Condition, Insurance & Upselling, etc.)
- **Sentiment** — positive | negative | mixed
- **Summarization** — kept for multi-task regularisation; inference uses mBART instead

Rare categories (Insurance & Upselling, Hidden Fees & Billing, etc.) are oversampled 3× so the model
learns them properly despite the class imbalance in the dataset.

> **Tip:** On Kaggle go to `Session options` and select **GPU T4 x2** for faster training.

In [ ]:
!pip install -q -U torchao transformers datasets peft accelerate huggingface_hub

In [ ]:
import time
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

## 0. Config
Mirrors `config.py`. Change these to control training behaviour.

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME   = "google/flan-t5-large"
ADAPTER_PATH = "miladghofrani/car-rental-peft-adapter-large"

# ── Dataset ────────────────────────────────────────────────────────────────
HF_DATASET = "miladghofrani/car-rental-reviews-short"

# ── Training ───────────────────────────────────────────────────────────────
MAX_STEPS     = None   # None = full training run
NUM_EPOCHS    = 5
LEARNING_RATE = 3e-4   # 1e-3 caused NaN gradients with fp16 — 3e-4 is stable

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Categories ─────────────────────────────────────────────────────────────
CATEGORIES = [
    "Cleanliness",
    "Vehicle Condition",
    "Pickup Experience",
    "Return Experience",
    "Hidden Fees & Billing",
    "Insurance & Upselling",
    "Staff & Communication",
    "Booking & App",
]

# Rare categories to oversample so flan-t5 learns them despite class imbalance.
# Insurance & Upselling appears ~10% of training data vs 56% for Staff & Communication.
OVERSAMPLE_CATS   = {"Insurance & Upselling", "Hidden Fees & Billing", "Cleanliness", "Booking & App"}
OVERSAMPLE_FACTOR = 3

## 1. Detect Device

In [ ]:
def get_device():
    print('🚀 Initializing the LLM Pipeline...')
    print('--------------------------------------------------')
    print('✅ All required libraries imported successfully.')
    print(f'✅ PyTorch version: {torch.__version__}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'✅ Processing device set to: {device.upper()}')
    print('--------------------------------------------------')
    return device

device = get_device()

## 2. Load & Build Multi-Task Dataset

Loads `miladghofrani/car-rental-reviews-short` from HF Hub and builds training examples per review:
- **Summarization**: review body → 1-sentence English summary
- **Classification**: review body → categories (with 3× oversampling for rare categories)
- **Sentiment**: review body → positive | negative | mixed

In [ ]:
from datasets import DatasetDict

def load_review_dataset():
    print(f"\n📥 Loading car rental reviews from HF Hub ({HF_DATASET})...")
    raw = load_dataset(HF_DATASET, split="train")
    dataset = DatasetDict({"train": raw})
    print(f"✅ Loaded {dataset['train'].num_rows:,} reviews")
    sample = dataset["train"][0]
    print("\n🔍 Sample:")
    print(f"  Review    : {sample['review_body'][:200]}...")
    print(f"  Summary   : {sample['summary']}")
    print(f"  Categories: {sample['categories']}")
    return dataset


def build_multitask_dataset(dataset):
    print(f"\n⚙️  Building multi-task examples...")
    categories_str = ", ".join(CATEGORIES)
    valid_sentiments = {"positive", "negative", "mixed"}
    inputs, outputs = [], []
    classification_inputs, classification_outputs = [], []

    for row in dataset["train"]:
        body       = (row.get("review_body") or "").strip()
        summary    = (row.get("summary") or "").strip()
        categories = row.get("categories") or []
        sentiment  = (row.get("sentiment") or "").strip().lower()

        if not body or not summary:
            continue

        # Summarization task
        inputs.append(f"Summarize the following car rental review.\n\n{body}\n\nSummary:")
        outputs.append(summary)

        # Classification task — with hint about rare categories
        cat_prompt = (
            f"Classify this car rental review into 1-3 of these categories: {categories_str}.\n"
            f"Use 'Insurance & Upselling' if insurance or extra products were pushed or required.\n"
            f"Use 'Hidden Fees & Billing' if unexpected charges or fees are mentioned.\n\n"
            f"Review: {body}\n\nCategories:"
        )
        cat_label = ", ".join(categories) if categories else "Staff & Communication"
        repeat = OVERSAMPLE_FACTOR if any(c in OVERSAMPLE_CATS for c in categories) else 1
        for _ in range(repeat):
            classification_inputs.append(cat_prompt)
            classification_outputs.append(cat_label)

        # Sentiment task
        if sentiment in valid_sentiments:
            inputs.append(
                f"What is the overall sentiment of this car rental review? "
                f"Answer with exactly one word: positive, negative, or mixed.\n\n"
                f"Review: {body}\n\nSentiment:"
            )
            outputs.append(sentiment)

    inputs.extend(classification_inputs)
    outputs.extend(classification_outputs)

    split = Dataset.from_dict({"input": inputs, "output": outputs}).train_test_split(test_size=0.1, seed=42)
    print(f"✅ {split['train'].num_rows:,} train / {split['test'].num_rows:,} validation examples")
    return split


raw_dataset       = load_review_dataset()
multitask_dataset = build_multitask_dataset(raw_dataset)

In [ ]:
# Verify training data before proceeding
print("Train examples:", multitask_dataset["train"].num_rows)
print("Test examples:", multitask_dataset["test"].num_rows)
print("\nSample input:")
print(multitask_dataset["train"][0]["input"])
print("\nSample output:")
print(multitask_dataset["train"][0]["output"])

assert multitask_dataset["train"].num_rows > 3000, "Dataset too small — check internet/dataset!"

import shutil, os
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/datasets"), ignore_errors=True)
print("\nCache cleared.")

## 3. Load Model & Tokenizer

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(
        f"📊 Model Parameter Report:\n"
        f"--------------------------------------------------\n"
        f"  Total Parameters:     {total:,}\n"
        f"  Trainable Parameters: {trainable:,}\n"
        f"  Percentage Trainable: {100 * trainable / total:.4f}%\n"
        f"--------------------------------------------------"
    )

def load_tokenizer(model_name=MODEL_NAME):
    print("\n🔤 Loading Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"✅ Tokenizer loaded for {model_name}.")
    return tokenizer

def load_llm_model(device, model_name=MODEL_NAME):
    print("\n🧠 Loading the LLM...")
    # T4 supports float16 (not bfloat16) — using torch_dtype halves VRAM vs float32
    target_dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=target_dtype).to(device)
    print(f"✅ {model_name} loaded using {target_dtype}.")
    return model

tokenizer      = load_tokenizer()
original_model = load_llm_model(device)
print_number_of_trainable_model_parameters(original_model)

## 4. Tokenize Dataset
Converts `input` / `output` columns into `input_ids` / `labels` for seq2seq training.

In [ ]:
def tokenize_dataset(tokenizer, dataset):
    print("\n⚙️  Tokenizing dataset for seq2seq training...")
    pad_id = tokenizer.pad_token_id

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input"], truncation=True, max_length=512
        )
        labels = tokenizer(
            batch["output"], truncation=True, max_length=128
        )
        # Replace padding in labels with -100 so the loss ignores them.
        # No padding here — DataCollatorForSeq2Seq pads dynamically per batch,
        # which is faster than padding everything to max_length upfront.
        model_inputs["labels"] = [
            [(t if t != pad_id else -100) for t in label]
            for label in labels["input_ids"]
        ]
        return model_inputs

    tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output"])
    print("✅ Tokenization complete.")
    return tokenized

tokenized_dataset = tokenize_dataset(tokenizer, multitask_dataset)

## 5. Inject LoRA Adapters

In [ ]:
def setup_peft_lora_model(original_model):
    print("\n🪄 Injecting LoRA Adapters (PEFT)...")
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=["q", "v"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    peft_model = get_peft_model(original_model, lora_config)
    print("✅ LoRA adapters injected.")
    print_number_of_trainable_model_parameters(peft_model)
    return peft_model

peft_model = setup_peft_lora_model(original_model)

## 6. Train & Save

> Set `MAX_STEPS = None` in the Config cell for a full training run (removes the 1-step dry-run limit).

In [ ]:
def train_and_save_peft_model(peft_model, tokenizer, tokenized_datasets):
    run_dir = f"./peft-training-run-{int(time.time())}"

    training_args = TrainingArguments(
        output_dir=run_dir,
        auto_find_batch_size=True,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=1,
        max_steps=MAX_STEPS if MAX_STEPS is not None else -1,
        gradient_checkpointing=True,
        # use_reentrant=False avoids needing enable_input_require_grads(),
        # which was accidentally saving embedding weights into the adapter checkpoint.
        gradient_checkpointing_kwargs={"use_reentrant": False},
        fp16=True,
        max_grad_norm=1.0,   # clip gradients to prevent fp16 overflow → NaN
    )

    mode = f"max_steps={MAX_STEPS}" if MAX_STEPS is not None else f"{NUM_EPOCHS} full epoch(s)"
    print(f"\n🏋️  Starting PEFT training ({mode})...")

    collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model, padding=True)

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        data_collator=collator,
    )

    trainer.train()

    # Sanity-check: catch NaN weights before saving
    import torch
    nan_found = False
    for name, param in peft_model.named_parameters():
        if 'lora' in name.lower() and torch.isnan(param).any():
            print(f"⚠️  NaN in {name}")
            nan_found = True
    if nan_found:
        raise RuntimeError("Training produced NaN weights — lower LEARNING_RATE or check data.")

    trainer.model.save_pretrained("./checkpoint-local")
    tokenizer.save_pretrained("./checkpoint-local")
    print("✅ Adapter weights saved to: ./checkpoint-local")


# No enable_input_require_grads() needed — handled by use_reentrant=False above
train_and_save_peft_model(peft_model, tokenizer, tokenized_dataset)
print("\n✅ Training complete!")

## 7. Push to HuggingFace Hub

Pushes both the dataset (one-time) and the trained adapter to your HF account.

**One-time setup:**
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → create a token with **write** permission
2. In Colab: click the 🔑 **Secrets** icon (left sidebar) → add secret named `HF_TOKEN` → paste your token

**First time only:** Run the "Push dataset" cell to upload `car_rental_reviews.jsonl` from your machine to HF Hub so future Colab sessions can load it.

In [ ]:
from huggingface_hub import login
import os

hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN", "")

login(token=hf_token)
print("✅ Logged in to HuggingFace Hub")

### 7a. Upload Dataset to HF Hub (run once from your local machine)

Run this cell **once from your local machine** (not Colab) to upload `car_rental_reviews.jsonl` to HF Hub.
After that, Colab always loads it from `miladghofrani/car-rental-reviews` in Section 2.

In [ ]:
# Run this once from your LOCAL MACHINE to push the dataset to HF Hub.
# After that you don't need to run it again — Section 2 loads from HF Hub automatically.
#
# From your terminal:
#   cd /Users/milad.ghofrani/P-Projects/generative-ai
#   python3 -c "
#   from datasets import load_dataset
#   ds = load_dataset('json', data_files='data_processing/car_rental_reviews.jsonl', split='train')
#   ds.push_to_hub('miladghofrani/car-rental-reviews')
#   print('Dataset pushed!')
#   "

# ── OR run this cell in Colab after uploading the file via Files panel ──────
# from google.colab import files
# uploaded = files.upload()  # select car_rental_reviews.jsonl
# from datasets import load_dataset
# ds = load_dataset("json", data_files="car_rental_reviews.jsonl", split="train")
# ds.push_to_hub("miladghofrani/car-rental-reviews")
# print(f"✅ Dataset pushed: {ds.num_rows:,} reviews")

### 7b. Push Trained Adapter to HF Hub

In [ ]:
from huggingface_hub import HfApi
import os

# Use upload_folder so this works even if peft_model is no longer in memory
# (e.g. running this cell alone after a background training run completed).
checkpoint = "./checkpoint-local"
assert os.path.isdir(checkpoint), f"Checkpoint not found at {checkpoint} — did training complete?"

HfApi().upload_folder(
    folder_path=checkpoint,
    repo_id=ADAPTER_PATH,
    repo_type="model",
)
print(f"✅ Adapter pushed to: https://huggingface.co/{ADAPTER_PATH}")